# 08 – Optimizacija Zaliha (EOQ kalkulator)

Na osnovu ML predikcija potražnje računamo optimalne parametre zaliha:

- **EOQ** (Economic Order Quantity) – optimalna količina narudžbe
- **Sigurnosna zaliha** (Safety Stock) – buffer za varijabilnost potražnje
- **ROP** (Reorder Point) – tačka ponovne narudžbe
- **ABC analiza** – klasifikacija proizvoda po vrijednosti potražnje
- **Narudžbenica** – generisanje preporuka za konkretnu prodavnicu
- **What-if analiza** – uticaj nivo usluge i lead time na troškove

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'
MODELS    = '../models/'

# ── Parametri (korisnik može mijenjati) ─────────────────────────────
S             = 50.0   # Trošak narudžbe (USD po narudžbi)
UNIT_COST     = 1.0    # Cijena po jedinici (relativna; skalirati po potrebi)
HOLDING_RATE  = 0.25   # Trošak čuvanja zaliha (25% godišnje od vrijednosti)
LEAD_TIME     = 7      # Lead time dobavljača (dani)
SERVICE_LEVEL = 0.95   # Nivo usluge (95%)
# ────────────────────────────────────────────────────────────────────

H = HOLDING_RATE * UNIT_COST          # godišnji trošak čuvanja po jed.
Z = stats.norm.ppf(SERVICE_LEVEL)     # z-score za nivo usluge

print(f'Parametri:')
print(f'  Trošak narudžbe (S):     {S:.2f} USD/narudžbi')
print(f'  Trošak čuvanja (H):      {H:.4f} USD/jed/god ({HOLDING_RATE*100:.0f}% godišnje)')
print(f'  Lead time:               {LEAD_TIME} dana')
print(f'  Nivo usluge:             {SERVICE_LEVEL*100:.0f}%  →  Z = {Z:.3f}')

## 1. Učitaj podatke i izračunaj statistike potražnje

In [ ]:
# Historijski podaci za statistike potražnje
train = pd.read_parquet(
    PROCESSED + 'train_features.parquet',
    columns=['date', 'store_nbr', 'family', 'sales']
)

# Ensemble predikcije (val set)
ens = pd.read_parquet(MODELS + 'ensemble_val_preds.parquet')

print(f'Historijski podaci: {train.shape}  ({train["date"].min().date()} – {train["date"].max().date()})')
print(f'Ensemble predikcije: {ens.shape}')

# Statistike dnevne potražnje po (store, family)
demand_stats = (
    train.groupby(['store_nbr', 'family'])['sales']
    .agg(mean_daily='mean', std_daily='std', n_days='count')
    .reset_index()
)
demand_stats['std_daily'] = demand_stats['std_daily'].fillna(0)
demand_stats['annual_demand'] = demand_stats['mean_daily'] * 365

print(f'\nBrojevi (store, family) parova: {len(demand_stats)}')
print(demand_stats.describe().round(2).to_string())

## 2. EOQ – Economic Order Quantity

In [ ]:
def compute_eoq(D, S, H):
    """EOQ = sqrt(2DS/H). Vraća 0 za nultu potražnju."""
    mask = (D > 0) & (H > 0)
    eoq  = np.where(mask, np.sqrt(2 * D * S / H), 0)
    return np.round(eoq, 1)

def compute_safety_stock(std_daily, lead_time, Z):
    """SS = Z * std_daily * sqrt(lead_time)"""
    return np.round(Z * std_daily * np.sqrt(lead_time), 1)

def compute_rop(mean_daily, lead_time, safety_stock):
    """ROP = mean_daily * lead_time + safety_stock"""
    return np.round(mean_daily * lead_time + safety_stock, 1)

def annual_cost(D, Q, S, H, SS):
    """TC = (D/Q)*S + (Q/2 + SS)*H"""
    safe_Q = np.where(Q > 0, Q, np.inf)
    return np.where(D > 0, (D / safe_Q) * S + (Q / 2 + SS) * H, 0)

inv = demand_stats.copy()
inv['EOQ']          = compute_eoq(inv['annual_demand'], S, H)
inv['safety_stock'] = compute_safety_stock(inv['std_daily'], LEAD_TIME, Z)
inv['ROP']          = compute_rop(inv['mean_daily'], LEAD_TIME, inv['safety_stock'])
inv['orders_year']  = np.where(inv['EOQ'] > 0, np.round(inv['annual_demand'] / inv['EOQ'], 1), 0)
inv['avg_inventory']= inv['EOQ'] / 2 + inv['safety_stock']
inv['annual_cost']  = annual_cost(inv['annual_demand'], inv['EOQ'], S, H, inv['safety_stock'])

# Samo aktivni proizvodi
active = inv[inv['mean_daily'] > 0].copy()
print(f'Aktivnih (store, family) parova: {len(active)} od {len(inv)}')
print(f'\nProsjek po aktivnom paru:')
print(f'  EOQ:            {active["EOQ"].mean():.1f} jedinica')
print(f'  Safety stock:   {active["safety_stock"].mean():.1f} jedinica')
print(f'  ROP:            {active["ROP"].mean():.1f} jedinica')
print(f'  Narudžbi/god.:  {active["orders_year"].mean():.1f}')
print(f'  Godišnji trošak:{active["annual_cost"].mean():.2f} USD')

## 3. ABC analiza – klasifikacija proizvoda

In [ ]:
# ABC po godišnjoj potražnji po kategoriji (aggregated across stores)
family_demand = (
    active.groupby('family')['annual_demand']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)
family_demand['cumulative_pct'] = family_demand['annual_demand'].cumsum() / family_demand['annual_demand'].sum() * 100

def abc_class(pct):
    if pct <= 80:   return 'A'
    elif pct <= 95: return 'B'
    else:           return 'C'

family_demand['ABC'] = family_demand['cumulative_pct'].apply(abc_class)

# Dodaj ABC na inv
active = active.merge(family_demand[['family', 'ABC']], on='family', how='left')

abc_summary = family_demand.groupby('ABC').agg(
    n_kategorija=('family', 'count'),
    udjel_potraznje=('annual_demand', lambda x: f"{x.sum()/family_demand['annual_demand'].sum()*100:.1f}%")
).reset_index()

print('ABC analiza po kategoriji proizvoda:')
print(abc_summary.to_string(index=False))
print()
print('A kategorije (80% potražnje):', family_demand[family_demand['ABC']=='A']['family'].tolist())

# Vizualizacija – Pareto
fig, ax1 = plt.subplots(figsize=(13, 5))
colors_abc = {'A': '#E53935', 'B': '#FB8C00', 'C': '#43A047'}
bar_colors = [colors_abc[c] for c in family_demand['ABC']]

ax1.bar(family_demand['family'], family_demand['annual_demand'], color=bar_colors, alpha=0.8)
ax1.set_ylabel('Godišnja potražnja')
ax1.set_xlabel('Kategorija')
plt.xticks(rotation=45, ha='right', fontsize=8)

ax2 = ax1.twinx()
ax2.plot(family_demand['family'], family_demand['cumulative_pct'], color='navy', marker='.', linewidth=2)
ax2.axhline(80, color='red',    linestyle='--', alpha=0.6, label='80% (A/B granica)')
ax2.axhline(95, color='orange', linestyle='--', alpha=0.6, label='95% (B/C granica)')
ax2.set_ylabel('Kumulativni udio (%)')
ax2.set_ylim(0, 105)
ax2.legend(loc='center right')

from matplotlib.patches import Patch
legend_abc = [Patch(facecolor=c, label=f'Klasa {k}') for k, c in colors_abc.items()]
ax1.legend(handles=legend_abc, loc='upper right')

plt.title('ABC analiza – Pareto dijagram potražnje po kategoriji')
plt.tight_layout()
plt.savefig(MODELS + 'inv_abc_pareto.png', dpi=120, bbox_inches='tight')
plt.show()

## 4. Vizualizacija EOQ distribucije

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col, label in zip(
    axes,
    ['EOQ', 'safety_stock', 'orders_year'],
    ['EOQ (jed.)', 'Sigurnosna zaliha (jed.)', 'Narudžbi/godišnje']
):
    data = active[col].clip(upper=active[col].quantile(0.99))
    ax.hist(data, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
    ax.axvline(active[col].median(), color='red', linestyle='--', label=f'Medijan: {active[col].median():.1f}')
    ax.set_xlabel(label)
    ax.set_ylabel('Frekvencija')
    ax.legend(fontsize=9)

plt.suptitle('Distribucija EOQ parametara (aktivni parovi)', fontsize=12)
plt.tight_layout()
plt.savefig(MODELS + 'inv_distributions.png', dpi=120, bbox_inches='tight')
plt.show()

## 5. Generisanje narudžbenice za konkretnu prodavnicu

In [ ]:
# ── Odaberi prodavnicu i datum ───────────────────────────────────────
TARGET_STORE   = 1
ORDER_DATE     = pd.Timestamp('2017-08-01')
CURRENT_STOCK  = None   # None = pretpostavi stock = 0 (konzervativno)
# ────────────────────────────────────────────────────────────────────

# Ensemble predikcija za narednih LEAD_TIME dana (period isporuke)
forecast_window = ens[
    (ens['store_nbr'] == TARGET_STORE) &
    (ens['date'] >= ORDER_DATE) &
    (ens['date'] <  ORDER_DATE + pd.Timedelta(days=LEAD_TIME))
].groupby('family')['ensemble_pred'].sum().reset_index()
forecast_window.rename(columns={'ensemble_pred': 'demand_lead_time'}, inplace=True)

# Spoji sa EOQ parametrima
store_inv = active[active['store_nbr'] == TARGET_STORE][[
    'family', 'ABC', 'mean_daily', 'EOQ', 'safety_stock', 'ROP',
    'orders_year', 'annual_cost'
]].merge(forecast_window, on='family', how='left')

store_inv['demand_lead_time'] = store_inv['demand_lead_time'].fillna(store_inv['mean_daily'] * LEAD_TIME)

# Trenutna zaliha (pretpostavka ako nije zadana)
if CURRENT_STOCK is None:
    store_inv['current_stock'] = store_inv['mean_daily'] * 3   # 3-dnevna zaliha
else:
    store_inv['current_stock'] = CURRENT_STOCK

# Da li treba naručiti? current_stock <= ROP
store_inv['needs_order'] = store_inv['current_stock'] <= store_inv['ROP']
store_inv['order_qty']   = np.where(store_inv['needs_order'], store_inv['EOQ'], 0)
store_inv['urgency']     = np.where(
    store_inv['current_stock'] < store_inv['demand_lead_time'], 'HITNO', 'Normalno'
)

orders = store_inv[store_inv['needs_order']].sort_values(['urgency', 'ABC', 'annual_cost'], ascending=[True, True, False])

print(f'Narudžbenica za Prodavnicu {TARGET_STORE} – {ORDER_DATE.date()}')
print(f'Lead time: {LEAD_TIME} dana | Nivo usluge: {SERVICE_LEVEL*100:.0f}%')
print(f'Ukupno kategorija: {len(store_inv)}')
print(f'Kategorija koje treba naručiti: {store_inv["needs_order"].sum()}')
print(f'Hitnih narudžbi: {(orders["urgency"]=="HITNO").sum()}')
print()
print(orders[['family','ABC','urgency','current_stock','ROP','order_qty','demand_lead_time']]
      .rename(columns={
          'family':'Kategorija','ABC':'Klasa','urgency':'Prioritet',
          'current_stock':'Zaliha','ROP':'ROP','order_qty':'Naruči (jed.)',
          'demand_lead_time':'Potražnja u LT'
      })
      .to_string(index=False))

In [ ]:
# Grafički prikaz narudžbenice
plot_df = orders.head(15).sort_values('order_qty', ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_urgency = {'HITNO': '#E53935', 'Normalno': '#1976D2'}
bar_clr = [colors_urgency[u] for u in plot_df['urgency']]

bars = ax.barh(plot_df['family'], plot_df['order_qty'], color=bar_clr, alpha=0.85)
ax.axvline(plot_df['order_qty'].mean(), color='black', linestyle='--', alpha=0.5, label='Prosjek EOQ')

for bar, abc in zip(bars, plot_df['ABC']):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            f'[{abc}]', va='center', fontsize=8)

from matplotlib.patches import Patch
leg = [Patch(facecolor=c, label=l) for l, c in colors_urgency.items()]
ax.legend(handles=leg)
ax.set_xlabel('Količina narudžbe (EOQ, jedinice)')
ax.set_title(f'Narudžbenica – Prodavnica {TARGET_STORE}, {ORDER_DATE.date()}')
plt.tight_layout()
plt.savefig(MODELS + 'inv_order_chart.png', dpi=120, bbox_inches='tight')
plt.show()

## 6. What-if: Uticaj nivoa usluge na sigurnosnu zalihu

In [ ]:
service_levels = [0.80, 0.85, 0.90, 0.95, 0.99]
z_scores       = [stats.norm.ppf(sl) for sl in service_levels]

# Ukupna sigurnosna zaliha po nivou usluge (suma svih aktivnih)
total_ss   = []
total_cost = []

for z in z_scores:
    ss   = compute_safety_stock(active['std_daily'], LEAD_TIME, z)
    cost = annual_cost(active['annual_demand'], active['EOQ'], S, H, ss)
    total_ss.append(ss.sum())
    total_cost.append(cost.sum())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sl_labels = [f'{int(sl*100)}%' for sl in service_levels]

axes[0].bar(sl_labels, total_ss, color='steelblue', alpha=0.85)
axes[0].set_xlabel('Nivo usluge')
axes[0].set_ylabel('Ukupna sigurnosna zaliha (jed.)')
axes[0].set_title('Sigurnosna zaliha vs. Nivo usluge')
for i, v in enumerate(total_ss):
    axes[0].text(i, v + max(total_ss)*0.01, f'{v:,.0f}', ha='center', fontsize=9)

axes[1].bar(sl_labels, total_cost, color='darkorange', alpha=0.85)
axes[1].set_xlabel('Nivo usluge')
axes[1].set_ylabel('Ukupni godišnji trošak (USD)')
axes[1].set_title('Godišnji trošak vs. Nivo usluge')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
for i, v in enumerate(total_cost):
    axes[1].text(i, v + max(total_cost)*0.01, f'{v:,.0f}', ha='center', fontsize=9)

plt.suptitle('What-if: Nivo usluge (svi parovi)', fontsize=12)
plt.tight_layout()
plt.savefig(MODELS + 'inv_whatif_service.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'{'Nivo usluge':>15} {'Z-score':>10} {'Ukupna SS':>15} {'God. trošak':>15}')
print('-' * 57)
for sl, z, ss, tc in zip(service_levels, z_scores, total_ss, total_cost):
    print(f'{sl*100:>14.0f}%  {z:>10.3f}  {ss:>15,.0f}  {tc:>15,.2f}')

## 7. What-if: Uticaj lead time na ROP

In [ ]:
lead_times  = [3, 5, 7, 10, 14, 21]
z_fixed     = stats.norm.ppf(SERVICE_LEVEL)

median_rop  = []
median_ss   = []

for lt in lead_times:
    ss  = compute_safety_stock(active['std_daily'], lt, z_fixed)
    rop = compute_rop(active['mean_daily'], lt, ss)
    median_ss.append(ss.median())
    median_rop.append(rop.median())

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(lead_times, median_rop, 'o-', color='steelblue',  label='Medijan ROP',            linewidth=2)
ax.plot(lead_times, median_ss,  's--',color='darkorange', label='Medijan sigurnosne zalihe', linewidth=2)
ax.set_xlabel('Lead time (dani)')
ax.set_ylabel('Jedinice')
ax.set_title(f'What-if: Lead time – nivo usluge {SERVICE_LEVEL*100:.0f}%')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(MODELS + 'inv_whatif_leadtime.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'{'Lead time':>12} {'Medijan SS':>14} {'Medijan ROP':>14}')
print('-' * 42)
for lt, ss, rop in zip(lead_times, median_ss, median_rop):
    print(f'{lt:>10} d  {ss:>14.1f}  {rop:>14.1f}')

## 8. Sačuvaj rezultate za web app

In [ ]:
# Sačuvaj EOQ parametre za sve parove
active[['store_nbr','family','ABC','mean_daily','std_daily','annual_demand',
        'EOQ','safety_stock','ROP','orders_year','avg_inventory','annual_cost']].to_parquet(
    MODELS + 'inventory_params.parquet', index=False
)

# Sačuvaj family-level ABC klasifikaciju
family_demand.to_parquet(MODELS + 'abc_classification.parquet', index=False)

# Sačuvaj parametre koji su korišćeni
import json
inv_config = {
    'ordering_cost':  S,
    'unit_cost':      UNIT_COST,
    'holding_rate':   HOLDING_RATE,
    'lead_time':      LEAD_TIME,
    'service_level':  SERVICE_LEVEL,
    'z_score':        round(Z, 4),
}
with open(MODELS + 'inventory_config.json', 'w') as f:
    json.dump(inv_config, f, indent=2)

print('Sačuvano:')
print(f'  inventory_params.parquet   – EOQ/SS/ROP za {len(active)} (store, family) parova')
print(f'  abc_classification.parquet – ABC klase za {len(family_demand)} kategorija')
print(f'  inventory_config.json      – parametri kalkulatora')
print()
print('Slike:')
print('  inv_abc_pareto.png, inv_distributions.png')
print('  inv_order_chart.png')
print('  inv_whatif_service.png, inv_whatif_leadtime.png')
print()
print(f'Ukupna godišnja procijenjena ušteda vs. bez SS:')
print(f'  Servisirane narudžbe (nivo {SERVICE_LEVEL*100:.0f}%): {active["annual_cost"].sum():,.0f} USD')